<div style="background-color:rgb(0, 55, 207); padding: 30px; border-radius: 20px; box-shadow: 0 4px 15px rgba(105, 195, 255, 0.3); color:rgb(187, 201, 248); font-family: 'Times New Roman', serif;">

<h1 style="text-align: center; font-size: 38px; color: white; font-weight: bold;">Digital Twin Integration</h1>

<h3 style="font-size: 22px; color: white; font-weight: bold;">Libraries</h3>

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import torch
import requests
import time
import os
import requests
import torch.nn as nn
import torch.nn.functional as F
from transformers import pipeline

<h3 style="font-size: 22px; color: white; font-weight: bold;">Model Architecture</h3>

In [ ]:
class SignLanguageLSTM(nn.Module):
    def __init__(self, num_classes, input_size=126, hidden_size=128, dropout=0.5):
        super(SignLanguageLSTM, self).__init__()

        # CNN
        self.conv1 = nn.Conv1d(in_channels=input_size, out_channels=128, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(128)
        self.conv2 = nn.Conv1d(in_channels=128, out_channels=128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        self.relu = nn.ReLU()
        self.dropout_conv = nn.Dropout(p=dropout)

        # LSTM
        self.lstm1 = nn.LSTM(input_size=128, hidden_size=hidden_size, batch_first=True, bidirectional=True)
        self.norm1 = nn.LayerNorm(hidden_size * 2)
        self.dropout_lstm = nn.Dropout(p=dropout)

        self.lstm2 = nn.LSTM(input_size=hidden_size * 2, hidden_size=hidden_size, batch_first=True, bidirectional=True)
        self.norm2 = nn.LayerNorm(hidden_size * 2)

        self.dropout_fc = nn.Dropout(p=dropout)
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1) 
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.dropout_conv(x)
        x = x.permute(0, 2, 1)
        x, _ = self.lstm1(x)
        x = self.norm1(x)
        x = self.dropout_lstm(x)

        x, _ = self.lstm2(x)
        x = self.norm2(x)

        x = torch.max(x, dim=1)[0] 
        x = self.dropout_fc(x)
        x = self.fc2(x)
        return x

<h3 style="font-size: 22px; color: white; font-weight: bold;">Configuration</h3>

In [ ]:
MODEL_WEIGHTS = './best_model/LSTM71.11.pth' # or the model you have trained
NUM_CLASSES = 90 # Adjust to your number of classes
# Adjust to your class names   
CLASS_NAMES = [
    'all', 'almost', 'approve', 'before', 'boss', 'break', 'business', 'busy', 'but', 'buy', 'can', 
    'change', 'clock', 'computer', 'deaf', 'decide', 'delay', 'different', 'discuss', 'drink', 
    'eat', 'email', 'evaluate', 'explain', 'family', 'fine', 'finish', 'forget', 'full', 'give', 
    'goal', 'have', 'hearing', 'help', 'how', 'idea', 'improve', 'inform', 'last', 'later', 'leader', 
    'like', 'manager', 'many', 'meet', 'meeting', 'money', 'month', 'need', 'no', 'now', 'office', 
    'paper', 'plan', 'policy', 'presentation', 'problem', 'professional', 'provide', 'responsibility', 
    'result', 'role', 'same', 'schedule', 'secretary', 'sell', 'show', 'sorry', 'study', 'support', 
    'table', 'take', 'team', 'time', 'trade', 'understand', 'vacation', 'visit', 'wait', 'want', 
    'week', 'what', 'who', 'why', 'with', 'work', 'workshop', 'year', 'yes', 'yesterday'
]

# MediaPipe setup for hand landmarks
mp_hands = mp.solutions.hands
hand_detector = mp_hands.Hands(static_image_mode=False, 
                               max_num_hands=2, 
                               model_complexity=1, 
                               min_detection_confidence=0.5, 
                               min_tracking_confidence=0.5)

<h3 style="font-size: 22px; color: white; font-weight: bold;">Model-GPU</h3>

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load LSTM Model
model = SignLanguageLSTM(num_classes=NUM_CLASSES)
model.load_state_dict(torch.load(MODEL_WEIGHTS, map_location=device))
model = model.to(device)
model.eval()


<h3 style="font-size: 22px; color: white; font-weight: bold;">Open Web Cam</h3>

In [5]:
def capture_sign_from_camera():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Error: Cannot access camera.")
        return None

    print("Press 's' to start recording the sign, 'e' to end recording, or 'q' to quit.")
    recorded_frames = []
    recording = False
    start_time = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.flip(frame, 1)

        if recording:
            recorded_frames.append(frame.copy())
            cv2.circle(frame, (30, 30), 10, (0, 0, 255), -1)
            cv2.putText(frame, "Recording...", (50, 35), cv2.FONT_HERSHEY_SIMPLEX,
                        0.8, (0, 0, 255), 2)
            if time.time() - start_time > 5:
                recording = False
                print("Auto-stopped recording after 5 seconds.")
                break
        else:
            cv2.putText(frame, "Press 's' to start recording a sign", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
            cv2.putText(frame, "Press 'q' to quit", (10, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        cv2.imshow("Camera", frame)
        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):
            cap.release()
            cv2.destroyWindow("Camera")
            return None
        if key == ord('s') and not recording:
            recording = True
            start_time = time.time()
        if key == ord('e') and recording:
            recording = False
            break

    cap.release()
    cv2.destroyWindow("Camera")
    return recorded_frames



<h3 style="font-size: 22px; color: white; font-weight: bold;">Translation (sign to text)</h3>


In [6]:
def predict_sign(frames):
    if not frames:
        return None

    all_landmarks = []
    for frame in frames:
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hand_detector.process(rgb_frame)

        hand_features = []
        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                single_hand = []
                for lm in hand_landmarks.landmark:
                    single_hand.extend([lm.x, lm.y, lm.z])
                hand_features.append(single_hand)

        if len(hand_features) == 1:
            hand_features.append([0.0] * 63)
        elif len(hand_features) == 0:
            hand_features = [[0.0] * 63, [0.0] * 63]

        features = np.array(hand_features[0] + hand_features[1])
        all_landmarks.append(features)

    data_tensor = torch.tensor(all_landmarks, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(data_tensor)
        pred_class = output.argmax(dim=1).item()

    predicted_text = CLASS_NAMES[pred_class]
    return predicted_text

<h3 style="font-size: 22px; color: white; font-weight: bold;">Words to Sentence (GenAI)</h3>


In [ ]:
# Load better text generation pipeline
nlp = pipeline("text2text-generation", model="google/flan-t5-large")

def generate_sentence_better(accepted_words):
    input_prompt = f"Create a meaningful English sentence using the following words: {', '.join(accepted_words)}."
    
    output = nlp(input_prompt, max_length=50, do_sample=False)[0]['generated_text']
    return output


<h3 style="font-size: 22px; color: white; font-weight: bold;">Avatar API call</h3>


In [7]:
def generate_avatar_video(text):
    if text is None:
        return None
    print(f"Sending text to Digital Twin API: '{text}'")
    try:
        response = requests.post("http://localhost:3000/generate", json={"text": text}, timeout=60)
        response.raise_for_status()
    except requests.RequestException as e:
        print("Error communicating with Digital Twin API:", e)
        return None
    data = response.json()
    video_url = data.get("videoUrl") or data.get("url")
    if video_url:
        print("Received video URL:", video_url)
    else:
        print("No video URL received. Response:", data)
    return video_url

<h3 style="font-size: 22px; color: white; font-weight: bold;">Show results</h3>


In [ ]:
def play_video_from_url(video_url):
    if not video_url:
        return

    try:
        print("Downloading avatar video...")
        video_data = requests.get(video_url, timeout=60).content
        video_path = "avatar_output.mp4"
        with open(video_path, "wb") as f:
            f.write(video_data)
        print(f"Video saved as {video_path}")
    except Exception as e:
        print("Failed to download video:", e)
        return

    while True:
        print("Opening avatar video with VLC...")
        os.system(f'start vlc --play-and-exit {video_path}')

        print("Press 'r' to replay, any other key to continue...")
        key = input().strip().lower()
        if key == 'r':
            continue
        else:
            break

<h3 style="font-size: 22px; color: white; font-weight: bold;">Main Code</h3>


In [ ]:
if __name__ == "__main__":
    print("Starting Sign Language Translation with Digital Twin Integration...")

    while True:
        accepted_words = []

        while True:
            frames = capture_sign_from_camera()  # Open, capture, close
            if frames is None or len(frames) == 0:
                print("No frames captured or quit requested.")
                break

            word = predict_sign(frames)
            print(f"Predicted word: {word}")

            decision = input("Accept word? [y = yes, n = no, f = finish]: ").strip().lower()

            if decision == 'y':
                accepted_words.append(word)
                print(f"✅ Added '{word}'.")
            elif decision == 'n':
                print("🔁 Retry recording.")
                continue
            elif decision == 'f':
                finish_decision = input(f"Do you want to add '{word}' before finishing? [y/n]: ").strip().lower()
                if finish_decision == 'y':
                    accepted_words.append(word)
                    print(f"✅ Added '{word}' before finishing.")
                else:
                    print("⏩ Word skipped before finishing.")
                break
            else:
                print("❓ Invalid input. Press 'y', 'n', or 'f'.")
                continue

        if not accepted_words:
            print("⚠️ No words collected. Starting over...")
            continue

        generated_sentence = generate_sentence_better(accepted_words) 
        print(f"📝 Generated sentence: {generated_sentence}")

        # Optional: Send to API
        video_url = generate_avatar_video(generated_sentence)
        if video_url:
            play_video_from_url(video_url)

        another = input("Start another translation? [y/n]: ").strip().lower()
        if another != 'y':
            print("👋 Exiting. Goodbye!")
            break

